# ESADE DSF26 — Personal Loan Credit Decision
## Full pipeline: forensic audit → cleaning → modeling → submission

**Goal:** predict `credit_decision` (1 = approved, 0 = rejected) for 5,000 May-2026 loan applications,
trained on 25,000 historical analyst decisions (2022–2026). **Metric: accuracy** on hard 0/1 labels.

This dataset was *deliberately* salted with traps ("assembled from multiple legacy source systems").
Our approach: never trust a column until it survives a train-vs-test integrity check.

### Summary of traps found & how we handled them

| # | Trap | Evidence | Action |
|---|------|----------|--------|
| 1 | `internal_code` is a **planted leak** | r = +0.92 with target in train, but decorrelated from every real risk signal in test (scrambled) | **Drop** |
| 2 | `external_pd_score` | 100% missing in test (5% in train) | **Drop** |
| 3 | `ann_income` mixed units | bimodal: half the values in thousands (gap 700–2000 has zero rows) | ×1000 below 700 |
| 4 | `prev_default` mixed encodings | `{0, No, 1, Yes, NaN}` | map to {0, 1, NaN} |
| 5 | `date` mixed formats | `2024-11-29` vs `Nov-2025` | parse both |
| 6 | `risk_indicator_1` ≡ `risk_indicator_2` | corr = 1.0 on overlapping rows | coalesce into one |
| 7 | `race` / `religion` in a credit model | +0.8pt accuracy from pure analyst bias; ~zero correlation with actual risk | **Exclude** (legal/ethical) |
| 8 | possible prompt injection in `analyst_opinion` | scanned all 60 unique values — clean template sentences | use as feature |

**Validation:** 5-fold out-of-fold CV **and** a time-based split (train ≤2024 → validate 2025/26),
because the test set is entirely May 2026 — a random split alone could hide temporal drift.


In [ ]:
import pandas as pd, numpy as np
import lightgbm as lgb, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
DATA = "../data/"
train = pd.read_csv(DATA + "train.csv")
test  = pd.read_csv(DATA + "test.csv")
y = train['credit_decision']
print(train.shape, test.shape)
print(y.value_counts(normalize=True).round(3))

## 1. Audit — missingness reveals the first trap

The target is nearly balanced (52/48), so a majority-class baseline only gets ~52% — every accuracy
point has to be earned.

Comparing missingness **between train and test** is the single most informative audit step:
a feature that exists in train but not in test is a trap for any model trained naively.

In [ ]:
miss = pd.DataFrame({'train%': (train.isna().mean()*100).round(1),
                     'test%':  (test.isna().mean()*100).round(1)})
miss['gap'] = (miss['test%'] - miss['train%']).round(1)
miss.sort_values('gap', ascending=False).head(8)

`external_pd_score` is **5% missing in train but 100% missing in test**.
A model that learns to rely on it performs great in cross-validation and gets nothing at prediction
time. **Dropped.**

## 2. The big one: `internal_code` is a planted leak

`internal_code` ("Internal processing code") correlates **+0.92** with the target — a simple
threshold on it scores 99% accuracy in train. Too good to be true. 

**Forensic test:** we need a signal that exists in *both* train and test and is strongly related to
the true decision. `analyst_opinion` is perfect: 60 template sentences whose train approval rates
range from 0.28 to 1.00, and the exact same 60 sentences appear in test. If `internal_code` is a real
risk variable, it must correlate with the opinion tier in test just like it does in train.

In [ ]:
print("simple rule internal_code>50, train accuracy:",
      ((train.internal_code > 50) == y).mean().round(4))

op_rate_map = train.groupby('analyst_opinion').credit_decision.mean()
tr_op = train.analyst_opinion.map(op_rate_map)
te_op = test.analyst_opinion.map(op_rate_map)

print("corr(internal_code, opinion approval rate) TRAIN:", train.internal_code.corr(tr_op).round(3))
print("corr(internal_code, opinion approval rate) TEST :", test.internal_code.corr(te_op).round(3))

**Train: +0.46. Test: ≈0.** The marginal distribution of `internal_code` is identical in
train and test (so a distribution check would *not* catch it), but in test it has been decoupled from
every genuine signal — i.e. **scrambled**. Anyone using it collapses to ~55% on the leaderboard.

We extended this *opinion-probe* to **every** feature: for each candidate, compare corr(feature,
opinion tier) in train vs test. Every other feature passed — `internal_code` was the only tampered column.

In [ ]:
def probe(col_tr, col_te, name):
    a, b = col_tr.corr(tr_op), col_te.corr(te_op)
    print(f"{name:18s} train {a:+.3f} | test {b:+.3f}" + ("   <-- TAMPERED" if abs(a)>.05 and abs(b)<abs(a)*.4 else ""))

probe(pd.to_numeric(train.ann_income), pd.to_numeric(test.ann_income), 'ann_income')
probe(train.amount, test.amount, 'amount')
probe(train.age, test.age, 'age')
probe(train.highest_ed, test.highest_ed, 'highest_ed')
probe(train.risk_indicator_3, test.risk_indicator_3, 'risk_indicator_3')
probe(train.internal_code, test.internal_code, 'internal_code')

## 3. Legacy-format cleaning

### 3a. `ann_income` — half the rows are in thousands
The log-histogram is bimodal with an empty gap between ~700 and ~2000: one legacy system stored
income in currency units (e.g. 16500), another in thousands (e.g. 76.4 = 76,400). We rescale
everything below 700 by ×1000. Same rule for `other_income` (nonzero values only).

In [ ]:
inc = pd.to_numeric(train.ann_income, errors='coerce')
counts, edges = np.histogram(np.log10(inc.clip(lower=0.1)), bins=12)
print("log10 bin edges:", edges.round(1))
print("counts:         ", counts, "   <- note the empty middle bins (the unit gap)")
print("share of rows < 700:", (inc < 700).mean().round(3))

### 3b. `date` — two formats, `prev_default` — two encodings
- dates: ISO `2024-11-29` and month-year `Nov-2025` (~50/50) → parse both, mid-month for the latter.
- `prev_default`: `{0, No}` → 0, `{1, Yes}` → 1, NaN stays NaN (LightGBM handles missing natively).
  Approval is 54% without prior default vs **~7%** with one — a powerful feature once unified.

### 3c. Risk indicators — duplicate columns + an inverted-U
- `risk_indicator_1` and `risk_indicator_2` have **correlation 1.0** on the 1,007 rows where both
  exist — same variable from two systems ("check consistency between variables"). → coalesce.
- Both the coalesced indicator and `risk_indicator_3` have an **inverted-U** relation with approval
  (mid-range ≈ 0.63 approval, top decile ≈ 0.26). Linear correlation looks tiny (-0.08) but the
  signal is real — so we add shape features: |risk − 55|, max, mean.

### 3d. Credit scores — three bureaus, one applicant
`cr_scores_fico/vantage/schufa` are 90–97% missing each (different scales, mostly mutually exclusive).
→ z-score each bureau on combined train+test, then coalesce into one `credit_z`.

In [ ]:
print("corr(r1, r2) where both present:",
      train.risk_indicator_1.corr(train.risk_indicator_2).round(3))
r12 = train.risk_indicator_1.fillna(train.risk_indicator_2)
print("\napproval rate by coalesced risk decile:")
print(train.groupby(pd.qcut(r12, 10), observed=True).credit_decision.mean().round(3))

## 4. Fairness — the ethics trap, quantified

`race` and `religion` are in the data, and the overview explicitly says to consider
"legal and regulatory limitations". Two facts from our audit:

1. Neither correlates with the *opinion tier* (actual risk) — corr ≈ 0.00.
2. Yet both shift approval rates (e.g. religion B: 55.9% vs C: 45.3%) — i.e. **pure analyst bias**.

Including them raises OOF accuracy from 0.857 to 0.864 (+0.8pt) — by reproducing discrimination.
We **exclude them** (along with `birth_year`-only signal beyond age, and `id`). This is the
defensible decision for the report: we measured the cost and rejected it knowingly.

## 5. Feature engineering — the full cleaning function

In [ ]:
def parse_dates(s):
    iso = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    mon = pd.to_datetime(s, format='%b-%Y', errors='coerce') + pd.Timedelta(days=14)
    return iso.fillna(mon)

def engineer(df):
    out = pd.DataFrame(index=df.index)
    d = parse_dates(df['date'])
    out['year'], out['month'] = d.dt.year, d.dt.month

    inc = pd.to_numeric(df['ann_income'], errors='coerce')
    out['ann_income'] = np.where(inc < 700, inc * 1000, inc)          # unit fix
    oth = pd.to_numeric(df['other_income'], errors='coerce')
    out['other_income'] = np.where((oth > 0) & (oth < 700), oth * 1000, oth)
    out['total_income'] = out['ann_income'] + out['other_income']
    out['amount'] = df['amount']
    out['amt_to_income'] = df['amount'] / (out['total_income'] + 1)   # affordability ratio
    out['log_income'] = np.log1p(out['total_income'])

    out['age'] = df['age'].fillna(out['year'] - df['birth_year'])     # impute age
    out['prev_default'] = df['prev_default'].map({'0':0,'No':0,'1':1,'Yes':1,0:0,1:1}).astype(float)
    out['highest_ed'], out['kids'] = df['highest_ed'], df['kids']
    out['vip'] = df['vip'].astype(str).map({'True':1,'False':0})

    out['risk12'] = df['risk_indicator_1'].fillna(df['risk_indicator_2'])  # r1 == r2
    out['risk3'] = df['risk_indicator_3']
    out['risk12_dev'] = (out['risk12'] - 55).abs()                    # inverted-U shape
    out['risk3_dev']  = (out['risk3'] - 55).abs()
    out['risk_max']  = out[['risk12','risk3']].max(axis=1)
    out['risk_mean'] = out[['risk12','risk3']].mean(axis=1)

    out['has_score'] = df[['cr_scores_fico','cr_scores_vantage','cr_scores_schufa']].notna().any(axis=1).astype(int)
    return out

X, Xte = engineer(train), engineer(test)

# credit scores: z-score per bureau on combined data, then coalesce
for c in ['cr_scores_fico','cr_scores_vantage','cr_scores_schufa']:
    allv = pd.concat([train[c], test[c]])
    mu, sd = allv.mean(), allv.std()
    X[c+'_z'], Xte[c+'_z'] = (train[c]-mu)/sd, (test[c]-mu)/sd
for df_ in (X, Xte):
    df_['credit_z'] = df_['cr_scores_fico_z'].fillna(df_['cr_scores_vantage_z']).fillna(df_['cr_scores_schufa_z'])
    df_.drop(columns=['cr_scores_fico_z','cr_scores_vantage_z','cr_scores_schufa_z'], inplace=True)

# categoricals (incl. the 60 analyst opinion templates — strongest legitimate feature)
for c in ['job_category', 'status', 'analyst_opinion']:
    cats = sorted(set(train[c].dropna()) | set(test[c].dropna()))
    X[c]   = pd.Categorical(train[c], categories=cats)
    Xte[c] = pd.Categorical(test[c],  categories=cats)

print(X.shape, "features:", list(X.columns))

**Excluded on purpose:** `internal_code` (scrambled leak), `external_pd_score` (absent in test),
`race`, `religion` (protected attributes), `id` (synthetic), `birth_year` (folded into `age`).

## 6. Validation strategy + single-model check

Two views, because the test set is entirely May 2026:
- **5-fold stratified OOF** — uses all data, gives the threshold-tuning curve.
- **Time split** (train ≤2024 → validate 2025/26) — catches temporal drift a random split would hide.

Since the metric is *accuracy on hard labels*, we tune the decision threshold on OOF probabilities
instead of defaulting to 0.5.

In [ ]:
params = dict(objective='binary', learning_rate=0.03, num_leaves=63, min_child_samples=40,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              n_estimators=3000, verbose=-1, seed=42)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))
oof = np.zeros(len(X))
for tr_i, va_i in folds:
    m = lgb.LGBMClassifier(**params)
    m.fit(X.iloc[tr_i], y.iloc[tr_i], eval_set=[(X.iloc[va_i], y.iloc[va_i])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
    oof[va_i] = m.predict_proba(X.iloc[va_i])[:, 1]

ths = np.linspace(0.35, 0.65, 301)
accs = [((oof > t) == y).mean() for t in ths]
print(f"single LGBM OOF acc @0.5 = {((oof>0.5)==y).mean():.4f}")
print(f"best threshold {ths[np.argmax(accs)]:.3f} -> OOF acc {max(accs):.4f}")

recent = X.year >= 2025
m2 = lgb.LGBMClassifier(**params)
m2.fit(X[~recent], y[~recent], eval_set=[(X[recent], y[recent])],
       callbacks=[lgb.early_stopping(150, verbose=False)])
p2 = m2.predict_proba(X[recent])[:, 1]
print(f"time-split acc (<=2024 -> 2025/26): {max(((p2>t)==y[recent]).mean() for t in ths):.4f}")

## 7. Hyperparameter search

We ran an 18-config random search (learning rate, leaves, min_child_samples, feature/bagging
fractions, L2) scored by 5-fold OOF accuracy, and kept the top-3 configs:

In [ ]:
BEST_PARAMS = [
  {
    "objective": "binary",
    "verbose": -1,
    "n_estimators": 4000,
    "learning_rate": 0.0206873441980807,
    "num_leaves": 31,
    "min_child_samples": 40,
    "feature_fraction": 0.8441709185745426,
    "bagging_fraction": 0.6210789150702444,
    "bagging_freq": 1,
    "reg_lambda": 1.5860985239773935
  },
  {
    "objective": "binary",
    "verbose": -1,
    "n_estimators": 4000,
    "learning_rate": 0.0268712644097486,
    "num_leaves": 63,
    "min_child_samples": 20,
    "feature_fraction": 0.8109671816250753,
    "bagging_fraction": 0.7907091140489139,
    "bagging_freq": 1,
    "reg_lambda": 0.250069564524446
  },
  {
    "objective": "binary",
    "verbose": -1,
    "n_estimators": 4000,
    "learning_rate": 0.015732064225516363,
    "num_leaves": 31,
    "min_child_samples": 20,
    "feature_fraction": 0.6886520608889237,
    "bagging_fraction": 0.7632087704997071,
    "bagging_freq": 1,
    "reg_lambda": 0.06406258420863976
  }
]
for p in BEST_PARAMS: print({k: (round(v,4) if isinstance(v,float) else v) for k,v in p.items() if k not in ('objective','verbose','n_estimators')})

## 8. Final model — 12-member ensemble → submission CSV

- 9 × LightGBM (top-3 configs × 3 seeds) + 3 × XGBoost (3 seeds), all 5-fold.
- Average the probabilities, tune one threshold on the ensemble OOF, apply it to the averaged
  test probabilities, write `submissions/sub_v2_ensemble.csv`.

(~5–10 min runtime.)

In [ ]:
SEEDS = [42, 7, 2026]
oof_e = np.zeros(len(X)); pte = np.zeros(len(Xte)); n_members = 0

for cfg in BEST_PARAMS:                                   # 9 LightGBM members
    for sd in SEEDS:
        for tr_i, va_i in folds:
            m = lgb.LGBMClassifier(**{**cfg, 'seed': sd})
            m.fit(X.iloc[tr_i], y.iloc[tr_i], eval_set=[(X.iloc[va_i], y.iloc[va_i])],
                  callbacks=[lgb.early_stopping(150, verbose=False)])
            oof_e[va_i] += m.predict_proba(X.iloc[va_i])[:, 1]
            pte += m.predict_proba(Xte)[:, 1] / len(folds)
        n_members += 1

for sd in SEEDS:                                          # 3 XGBoost members
    for tr_i, va_i in folds:
        m = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.025, max_depth=6,
                              min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                              reg_lambda=1.0, enable_categorical=True, tree_method='hist',
                              early_stopping_rounds=150, eval_metric='logloss', seed=sd, verbosity=0)
        m.fit(X.iloc[tr_i], y.iloc[tr_i], eval_set=[(X.iloc[va_i], y.iloc[va_i])], verbose=False)
        oof_e[va_i] += m.predict_proba(X.iloc[va_i])[:, 1]
        pte += m.predict_proba(Xte)[:, 1] / len(folds)
    n_members += 1

oof_e /= n_members; pte /= n_members
accs = [((oof_e > t) == y).mean() for t in np.linspace(0.35, 0.65, 601)]
best_t = np.linspace(0.35, 0.65, 601)[int(np.argmax(accs))]
print(f"ensemble ({n_members} members) OOF acc: {max(accs):.4f} at threshold {best_t:.3f}")

submission = pd.DataFrame({'id': test['id'], 'credit_decision': (pte > best_t).astype(int)})
submission.to_csv("../submissions/sub_v2_ensemble.csv", index=False)
print("saved ../submissions/sub_v2_ensemble.csv")
print("predicted approval rate:", submission.credit_decision.mean().round(3),
      "| train base rate:", y.mean().round(3))

## 9. Conclusions

| What | Score |
|------|-------|
| Majority-class baseline | 0.520 |
| Opinion tiers alone | ~0.74 |
| Single tuned LightGBM (OOF) | ~0.858 |
| 12-member ensemble (OOF) | see above |
| Current leaderboard top | 0.8565 |

**Why this should hold on the private leaderboard:**
1. Every feature passed a train-vs-test integrity probe — we are not leaning on anything scrambled or absent in test.
2. Time-split validation ≈ OOF validation → no harmful temporal drift.
3. Threshold tuned on OOF probabilities (25k rows), not on public-leaderboard feedback → no p-hacking.
4. Protected attributes excluded by design — measured cost ≈ 0.8pt, knowingly rejected (see report).


## 10. Beyond the first ensemble — what worked and what didn't (v4–v8)

After the first ensemble (public 0.8520) we ran a sequence of controlled experiments. Every candidate
was evaluated on a **rolling next-month harness**: for each of the last 6 months in train, fit on all
data before that month and predict it (2,886 pooled out-of-time predictions) — the closest possible
simulation of the real task (train ends 2026-04-30; test is exactly May 2026).

| Experiment | Rolling / OOF result | Verdict |
|---|---|---|
| CatBoost as third model family | 0.8607 OOF alone — strongest single family | **kept** (got 60–100% of blend weight) |
| Blend-weight grid search (LGBM/XGB/CAT) | 0.8610 → v4, public 0.8560 | **kept** |
| CatBoost hyperparameter search (7 configs) | all within 0.2pt of default | dead end |
| 9-member CatBoost pool + stacking test | blend 0.8615 > LR stack 0.8606 → v6, public **0.8565** | **kept** |
| Recency-weighted training | 0.8600 vs baseline 0.8631 | hurts |
| Dropping time features | 0.8604 | hurts |
| Provenance features (source-system flags from the legacy formats) | 0.8600 vs 0.8604 | a wash — the format mess was only a cleaning test |
| Repeated CV (3 fold-seeds) | 0.8618 OOF → v7 | kept (best probabilities) |
| **Pseudo-labeling** test rows at ≥0.97 confidence, half weight | **+0.14pt** on rolling harness → v8 | **kept** |
| race/religion | +0.7pt | **refused** — protected attributes (see §4) |

### The threshold lesson
Because the metric is accuracy on hard labels, the probability→label cutoff matters as much as the
model. Submissions with predicted approval rates near **0.51** (matching the declining 2026 trend)
consistently beat looser cutoffs: v3 (approval 0.518) and v7 (0.524) both underperformed their
sibling files with ~0.512. v8 lands at 0.514.

### Leaderboard noise — why we stopped submitting
The public leaderboard uses 40% of the test set (2,000 rows; 1 row = 0.0005). Our last three
submissions differ pairwise by ~50 rows, of which the public split sees ~20; the paired noise on
that comparison is ±2–3 rows — the same magnitude as the observed score differences
(0.8550 / 0.8555 / 0.8565). The public board cannot resolve differences this small. We therefore
selected our two final submissions by validation (v8: best honest estimate; v6: public-best and
threshold-proven), not by public score — only the private 60% counts, and chasing the public split
is exactly the p-hacking the organizers warned about.

### Final methodology (v8)
1. Cleaning + features as in §5 (traps removed, protected attributes excluded).
2. CatBoost pool: 3 configs × 2 seeds × 3 CV fold-seeds (18 members, 5-fold each).
3. Semi-supervised: test rows with ensemble confidence ≥ 0.97 (~34%) added to every training fold
   as pseudo-labeled examples at half sample-weight (+0.14pt on the rolling harness).
4. Threshold tuned on OOF probabilities → approval rate 0.514, consistent with the 2026 trend.
